In [22]:
from oo_cqed_rhf import CQEDRHFCalculator
import numpy as np
import psi4
psi4.core.be_quiet()

In [38]:
def bfgs_update(xk, gk, xkp1, gkp1, Hk):
    """
    Performs a BFGS update of the Hessian approximation.  Needs two solution vectors (xk and xk+1)
    and their corresponding gradients (gk and gk+1) and the current Hessian approximation (Hk).

    Arguments
    ---------
    xk : numpy matrix representation of a row vector (1x2 matrix)
       the previous solution vector

    gk : numpy matrix representation of a row vector (1x2 matrix)
       the gradient at the previous solution vector

    xkp1 : numpy matrix representation of a row vector (1x2 matrix)
       the next solution vector

    gkp1 : numpy matrix representation of a row vector (1x2 matrix)
       the gradient at the next solution vector

    Hk : numpy matrix representatio of the Hessian (2x2 matrix)
       the current Hessian approximation


    Returns
    --------
    Hkp1 : numpy matrix representatio of the Hessian (2x2 matrix)
       the updated Hessian approximation
    """

    # compute yk
    yk = np.matrix(gkp1 - gk).T

    # compute sk
    sk = np.matrix(xkp1 - xk).T

    # compute alpha denominator for Hessian update
    a_d = yk.T @ sk

    # treat a_d as a scalar
    alpha = 1 / a_d[0,0]


    # compute beta denominator for Hessian update
    b_d = sk.T @ Hk @ sk

    # treat b_d as a scalr
    beta = - 1 / b_d[0,0]


    # Check if the dot product of alpha are beta are close to zero - these will make the Hessian update large
    if np.isclose(alpha, 0):
      print("Error: The dot product of y and s is close to zero. BFGS update is not performed.")
      return Hk

    if np.isclose(beta, 0):
      print("Error: The dot product of s and Hk and sk is close to zero. BFGS update is not performed.")
      return Hk

    # first part of update
    B1 = alpha * yk @ yk.T

    # second part of update
    B2 = beta * Hk @ sk @ sk.T @ Hk.T

    # update to Hessian
    Hkp1 = Hk + B1 + B2

    return Hkp1

def geom_bohr_to_angstrom_string(geom_bohr: np.ndarray,
                                 symbols: list[str]) -> str:
    """
    Convert an (N,3) array in Bohr to a psi4 geometry input string in Angstrom,
    appending the fixed Psi4 directives at the end.

    Returns a string like:
        O   x   y   z
        H   x   y   z
        -1 1
        no_reorient
        nocom
        symmetry c1
    """
    # Conversion factor: 1 bohr = 0.529177210903 Å
    bohr2ang = 0.529177210903

    # Convert coordinates
    geom_ang = geom_bohr * bohr2ang

    # Format atomic lines
    lines = []
    for sym, (x, y, z) in zip(symbols, geom_ang):
        lines.append(f"{sym:2s}  {x:14.12f}  {y:14.12f}  {z:14.12f}")

    # Append fixed Psi4 directives
    suffix = """
-1 1
no_reorient
nocom
symmetry c1
""".strip()

    # Combine and return full input block
    return "\n".join(lines + [suffix])


def optimize_geometry(calc,
                      x0_bohr: np.ndarray,      # shape (N_atoms,3)
                      symbols: list[str],
                      bfgs_update,
                      tol: float = 1e-6,
                      max_iter: int = 50):
    """
    BFGS‐style geometry optimizer using your calc.calc_force_and_energy().
    Returns the converged x (Bohr coords), E, gradient, and Hessian.
    """
    n_atoms = x0_bohr.shape[0]
    # total degrees of freedom
    ndim = 3 * n_atoms

    # flatten the initial geometry
    xk = x0_bohr.reshape(ndim)

    # initial Hessian guess (identity)
    Hk = np.eye(ndim)

    # initial energy & gradient
    geom_block = geom_bohr_to_angstrom_string(xk.reshape(n_atoms,3), symbols)
    Ek, gk_full = calc.calc_force_and_energy(
        geom_block, use_psi4_scf_grad=False
    )
    # flatten gradient
    gk = gk_full.reshape(ndim)

    converged = False
    for iteration in range(1, max_iter+1):
        # 1) Compute step pk = - Hk^{-1} gk
        pk = -np.linalg.solve(Hk, gk)

        # 2) Update geometry
        xkp1 = xk + pk

        # 3) Evaluate energy & gradient at new x
        geom_block = geom_bohr_to_angstrom_string(
            xkp1.reshape(n_atoms,3), symbols
        )
        Ekp1, gkp1_full = calc.calc_force_and_energy(
            geom_block, use_psi4_scf_grad=False
        )
        gkp1 = gkp1_full.reshape(ndim)

        # 4) Check convergence
        norm_g = np.linalg.norm(gkp1)
        print(f"Iter {iteration:2d}: E = {Ekp1:.8f} Eh   ‖g‖ = {norm_g:.2e}")
        if norm_g < tol:
            converged = True
            break

        # 5) BFGS update of H
        #    we assume bfgs_update accepts flattened vectors
        Hkp1 = bfgs_update(xk, gk, xkp1, gkp1, Hk)

        # 6) Shift these for next iter
        xk, gk, Hk, Ek = xkp1, gkp1, Hkp1, Ekp1

    # end for

    # Final report
    if converged:
        print("\nConverged!")
    else:
        print("\nWARNING: max_iter reached without convergence.")

    final_geom = geom_bohr_to_angstrom_string(xk.reshape(n_atoms,3), symbols)
    print(f"Final geometry (Bohr):\n{xk.reshape(n_atoms,3)}")
    print(f"Final energy: {Ekp1:.8f} Eh")
    print(f"Final gradient:\n{gkp1_full}")
    print("Final Geometry String:\n")
    print(final_geom)
    return xk.reshape(n_atoms,3), Ekp1, gkp1_full, Hk


In [43]:


# lambda vector along z
lambda_vector = np.array([0, 0.1, 0.1])


# psi4 options
psi4_options = {
    "basis": "6-31++G**",
    "save_jk": True,
    "scf_type": "pk",
    "e_convergence": 1e-12,
    "d_convergence": 1e-12,
}


psi4.set_options(psi4_options)

## forward displaced geometry string
mol_string = """
    O            0.000000000000     0.000000000000    -0.023958388679
    H            0.000000000000     0.000000000000     0.923958388679
-1 1
no_reorient
nocom
symmetry c1
"""

mol = psi4.geometry(mol_string)

print(mol.geometry().to_array())
x0_bohr = mol.geometry().to_array()

# get atomic symbols from geometry
symbols = [mol.symbol(i) for i in range(mol.natom())]  # ["O","H"]






[[ 0.00000000  0.00000000 -0.04527479]
 [ 0.00000000  0.00000000  1.74602831]]


In [44]:
# we will pass our desired geometry string when we want to compute the gradient
calc = CQEDRHFCalculator(lambda_vector, mol_string, psi4_options)

# calculate the CQED-RHF energy and gradient at h2o_string_b, use our routines for all terms
qed_rhf_energy, qed_rhf_grad = calc.calc_force_and_energy(mol_string, use_psi4_scf_grad=False)


In [45]:
print(qed_rhf_grad)

[[-0.00000000 -0.00242250 -0.01290205]
 [ 0.00000000  0.00242250  0.01290205]]


In [46]:
optimize_geometry(calc, x0_bohr, symbols, bfgs_update, tol=1e-7, max_iter=50)



Iter  1: E = -75.33229134 Eh   ‖g‖ = 5.59e-03
Iter  2: E = -75.33230796 Eh   ‖g‖ = 3.18e-03
Iter  3: E = -75.33232973 Eh   ‖g‖ = 4.10e-03
Iter  4: E = -75.33241052 Eh   ‖g‖ = 1.14e-02
Iter  5: E = -75.33250415 Eh   ‖g‖ = 1.86e-02
Iter  6: E = -75.33265105 Eh   ‖g‖ = 2.20e-02
Iter  7: E = -75.33300748 Eh   ‖g‖ = 2.21e-02
Iter  8: E = -75.33348050 Eh   ‖g‖ = 8.16e-03
Iter  9: E = -75.33344798 Eh   ‖g‖ = 3.47e-02
Iter 10: E = -75.33386468 Eh   ‖g‖ = 2.70e-03
Iter 11: E = -75.33402465 Eh   ‖g‖ = 3.15e-03
Iter 12: E = -75.33215309 Eh   ‖g‖ = 6.52e-02
Iter 13: E = -75.33410486 Eh   ‖g‖ = 1.82e-03
Iter 14: E = -75.33415864 Eh   ‖g‖ = 1.61e-03
Iter 15: E = -75.33265983 Eh   ‖g‖ = 5.81e-02
Iter 16: E = -75.33421054 Eh   ‖g‖ = 2.16e-03
Iter 17: E = -75.33424906 Eh   ‖g‖ = 1.72e-03
Iter 18: E = -75.33328694 Eh   ‖g‖ = 4.72e-02
Iter 19: E = -75.33428562 Eh   ‖g‖ = 1.77e-03
Iter 20: E = -75.33431216 Eh   ‖g‖ = 1.19e-03
Iter 21: E = -75.33414912 Eh   ‖g‖ = 2.32e-02
Iter 22: E = -75.33434595 Eh   ‖g‖

(array([[ 0.00000000,  0.63135490,  0.21902605],
        [-0.00000000, -0.63135490,  1.48172746]]),
 -75.33439296054661,
 array([[-0.00000000, -0.00000000,  0.00000000],
        [ 0.00000000,  0.00000000, -0.00000000]]),
 matrix([[ 1.00000000, -0.00000000,  0.00000000,  0.00000000,  0.00000000,
          -0.00000000],
         [-0.00000000,  0.80182088, -0.29906640,  0.00000000,  0.19817912,
           0.29906640],
         [ 0.00000000, -0.29906640,  0.80192667, -0.00000000,  0.29906639,
           0.19807333],
         [ 0.00000000,  0.00000000, -0.00000000,  1.00000000, -0.00000000,
           0.00000000],
         [ 0.00000000,  0.19817912,  0.29906639, -0.00000000,  0.80182087,
          -0.29906640],
         [-0.00000000,  0.29906640,  0.19807333,  0.00000000, -0.29906640,
           0.80192668]]))

In [29]:
# initial step for BFGS update

# initialize Hessian
n_atoms = 2

Hi = np.eye(3 * n_atoms)

# initialize gradient in atomic units
gi = qed_rhf_grad

# initialize energy
Ei = qed_rhf_energy

# initial update in atomic units
pi = -np.linalg.inv(Hi) @ gi

# initial geometry in atomic units
xi = mol.geometry().to_array()

# update geometry
x = xi + pi

# update mol_string and get new gradient
mol_string = geom_bohr_to_angstrom_string(x, symbols)

# calculate the CQED-RHF energy and gradient at h2o_string_b, use our routines for all terms
qed_rhf_energy, qed_rhf_grad = calc.calc_force_and_energy(mol_string, use_psi4_scf_grad=False)

# get new gradient and energy
g = qed_rhf_grad
E = qed_rhf_energy


# prepare for BFGS updates
xk = np.copy(xi)
xkp1 = np.copy(x)
gk = np.copy(gi)
gkp1 = np.copy(g)
Hk = np.copy(Hi)


converged = False
for i in range(10):
    Hkp1 = bfgs_update(xk, gk, xkp1, gkp1, Hk)
    
    #print(F"Updated Hessian is")
    #print(Hkp1)
    
    # update solutions xk and xk+1
    xk = np.copy(xkp1)
    
    #print(F"solution is {xk}")
    
    pk = -np.linalg.inv(Hkp1) @ gkp1
    
    #print(F"Update is {pk}")
    
    xkp1 = xk + pk
    print(F"Updated solution is {xkp1[0,:]}")
    
    # update gradients gk and gk+1
    gk = np.copy(gkp1)
    # update mol_string and get new gradient
    mol_string = geom_bohr_to_angstrom_string(xkp1, symbols)
    
    # calculate the CQED-RHF energy and gradient at h2o_string_b, use our routines for all terms
    qed_rhf_energy, qed_rhf_grad = calc.calc_force_and_energy(mol_string, use_psi4_scf_grad=False)
    
    # get new gradient and energy
    gkp1 = qed_rhf_grad
    Ekp1 = qed_rhf_energy

    
    # compute norm of gradient
    norm_grad = np.linalg.norm(gkp1)
    print(F"Norm of gradient is {norm_grad}")
    if norm_grad < 1e-6:
        converged = True
        break
    else:
        Hk = np.copy(Hkp1)
    
    if converged:
        print("\n\n\nConverged!!!!")
        print(F"Final solution is {xkp1[0,:]}")
        print(F"Final gradient is {gkp1}")
        print(F"Final Hessian is")
        print(Hk)
    else:
        print("\n\n\nDid not converge")
        print(F"Final solution is {xkp1[0,:]}")
        print(F"Final gradient is {gkp1}")
        print(F"Final Hessian is")
        print(Hk)


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 6)